# Разбор A. Неопределённость и отказ от неподтверждённого переноса

Продолжаем сохранённый QSAR A2A. Сначала различите: 90% условный conformal-интервал, наблюдаемое покрытие на тесте и гарантию для нового химического домена. Для calibration использован максимум абсолютного остатка внутри каждого каркаса. Квантиль с конечновыборочным рангом ceil((m+1)·0.9) даёт условие группового покрытия только при обменности новых каркасов с calibration; это не гарантия для произвольного лекарства.

In [2]:
from pathlib import Path
import sys,json,csv,hashlib,math
import numpy as np
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'programm/course.json').is_file())
sys.path.insert(0,str(ROOT/'practicuum/shared'))
from course_support import DATA,workspace,write_json,init_dossier,compound
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
OUT=workspace('M12-L02','case')
QDATA=DATA/'target_qsar'
manifest=json.loads((QDATA/'manifest.json').read_text(encoding='utf-8'))
for name,digest in manifest['files_sha256'].items():
    assert hashlib.sha256((QDATA/name).read_bytes()).hexdigest()==digest,name
def read_json(name):return json.loads((QDATA/name).read_text(encoding='utf-8'))
def read_csv(name):
    with (QDATA/name).open(encoding='utf-8',newline='') as f:return list(csv.DictReader(f))
def publish_plot(fig,name):
    fig.savefig(OUT/name,dpi=150,bbox_inches='tight');display(fig);plt.close(fig)


После средней тестовой ошибки я хочу принять решение для конкретного кандидата. Здесь возникают два разных вопроса: какое покрытие показывают интервалы на отложенных данных и насколько кандидат похож на область обучения. Я проверяю оба, потому что хорошее покрытие в одном наборе не устанавливает его сохранение в новом химическом домене.

В сохранённом групповом conformal-протоколе calibration-оценка каркаса — максимум абсолютного остатка его соединений. Это выбор одновременного требования к группе. В условном каркасе из десяти молекул попадание девяти в интервалы даёт 90% по точкам этого каркаса, но условие «попали все» для него не выполнено. Поэтому код вычисляет два показателя и не подменяет один другим.

In [3]:
rows=read_csv('a2a_curated.csv');split=read_json('a2a_split.json');report=read_json('a2a_result.json')
heldout=read_csv('a2a_heldout_predictions.csv')
y=np.array([float(r['y']) for r in rows])
ids={k:np.array(v['row_ids'],dtype=int) for k,v in split.items()}
assert sorted(np.concatenate(list(ids.values())).tolist())==list(range(len(rows)))
for a,b in [('training','calibration'),('training','test'),('calibration','test')]:
    assert not (set(split[a]['scaffolds']) & set(split[b]['scaffolds']))
assert [int(r['row_id']) for r in heldout]==ids['test'].tolist()
observed=np.array([float(r['observed']) for r in heldout])
predicted=np.array([float(r['predicted']) for r in heldout])
assert np.array_equal(observed,y[ids['test']])


Проверим покрытие по соединениям и одновременное покрытие всех соединений каждого тестового каркаса. Нельзя объявлять первое вторым.

In [4]:
queries=report['query_results']
width=(queries[0]['conditional_90_interval'][1]-queries[0]['conditional_90_interval'][0])/2
absolute=abs(predicted-observed);groups=np.array([r['scaffold'] for r in heldout])
point_coverage=float(np.mean(absolute<=width))
group_coverage=float(np.mean([np.all(absolute[groups==g]<=width) for g in np.unique(groups)]))
print({'half_width_pKi':width,'point_coverage':point_coverage,'simultaneous_scaffold_coverage':group_coverage})
audit=read_json('a2a_audit.json')
assert math.isclose(group_coverage,audit['observed_test_scaffold_simultaneous_coverage'],abs_tol=1e-12)


{'half_width_pKi': 1.5003056953341032, 'point_coverage': 0.9543325526932084, 'simultaneous_scaffold_coverage': 0.9118457300275482}


Рассчитаем Morgan-сходство запросов только с настоящим training. Murcko-ядро группирует стереоизомеры вместе, а fingerprints сохраняют chirality. Идентичный граф или каркас запроса в train/calibration нарушил бы выбранный эксперимент. Для ациклических структур pipeline использует generic full topology вместо единой пустой scaffold-группы.

In [5]:
from rdkit import Chem,DataStructs
from rdkit.Chem import rdFingerprintGenerator
from rdkit.Chem.Scaffolds import MurckoScaffold
def scaffold(m):
    core=MurckoScaffold.GetScaffoldForMol(m)
    if core.GetNumAtoms():return 'MURCKO:'+Chem.MolToSmiles(core,isomericSmiles=False)
    return 'ACYCLIC_TOPOLOGY:'+Chem.MolToSmiles(MurckoScaffold.MakeScaffoldGeneric(Chem.RemoveHs(m)),isomericSmiles=False)
gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=2048,includeChirality=True)
fps=[gen.GetFingerprint(Chem.MolFromSmiles(rows[i]['smiles'])) for i in ids['training']]
nearest=[]
for q in queries:
    mol=Chem.MolFromSmiles(q['smiles']);g=scaffold(mol)
    assert g not in set(split['training']['scaffolds'])|set(split['calibration']['scaffolds'])
    similarities=DataStructs.BulkTanimotoSimilarity(gen.GetFingerprint(mol),fps)
    j=int(np.argmax(similarities));value=float(similarities[j])
    assert math.isclose(value,q['nearest_train_Tanimoto'],abs_tol=1e-12)
    nearest.append({'name':q['name'],'nearest_training_row_id':int(ids['training'][j]),'Tanimoto':value,
                    'predicted_pKi':q['predicted_pKi'],'conditional_90_interval':q['conditional_90_interval'],
                    'decision':'REVIEW: external domain evidence required'})
display(nearest)
fig,ax=plt.subplots()
ax.bar([r['name'] for r in nearest],[r['Tanimoto'] for r in nearest]);ax.set(ylim=(0,1),ylabel='Nearest training Morgan Tanimoto')
publish_plot(fig,'chemical_domain.png')
result={'kind':'calculated','scope':'audit of frozen QSAR predictions and chemical domain','nearest':nearest,
        'point_coverage':point_coverage,'simultaneous_scaffold_coverage':group_coverage,'new_training':'NOT_RUN','prospective_validation':'NOT_RUN'}


[{'name': 'istradefylline',
  'nearest_training_row_id': 1905,
  'Tanimoto': 0.373134328358209,
  'predicted_pKi': 6.033947221496264,
  'conditional_90_interval': [4.533641526162161, 7.534252916830368],
  'decision': 'REVIEW: external domain evidence required'},
 {'name': 'caffeine',
  'nearest_training_row_id': 2418,
  'Tanimoto': 0.3404255319148936,
  'predicted_pKi': 6.277793943980605,
  'conditional_90_interval': [4.777488248646502, 7.778099639314708],
  'decision': 'REVIEW: external domain evidence required'}]

<Figure size 640x480 with 1 Axes>

Максимальное сходство истрадефиллина лишь ≈0.373, кофеина ≈0.340. Универсального доказанного порога здесь нет. Интервал истрадефиллина примерно 4.53–7.53 pKi широк; похожесть не доказывает применимость. Предсказанная активность не является измерением, механизмом, экспозицией или клиническим эффектом. Решение — запросить согласованные независимые assay для релевантного химического ряда; не подбирать порог после просмотра запроса.

Далее я сравниваю запрос только с настоящим training: если включить test или calibration в банк ближайших соседей, описание области обучения уже будет другим. Максимальное сходство и отсутствие каркаса дают диагностические сведения, но не вероятность правильного прогноза. Для низкоподдержанного кандидата рабочим решением может стать запрос независимого измерения и ограничение использования числа. Предпосылки обменности групп и сопоставимости assay должны обсуждаться отдельно от красивой ширины интервала.

In [6]:
write_json(OUT/'result.json',result)
from dossier_records import append_record
print(append_record(init_dossier(),'M12-L02','A',result,ROOT,inputs=['practicuum/shared/data/target_qsar/a2a_curated.csv', 'practicuum/shared/data/target_qsar/a2a_dataset.json', 'practicuum/shared/data/target_qsar/a2a_result.json', 'practicuum/shared/data/target_qsar/a2a_split.json', 'practicuum/shared/data/target_qsar/a2a_heldout_predictions.csv', 'practicuum/shared/data/target_qsar/a2a_audit.json', 'practicuum/shared/data/target_qsar/hmgcr_curated.csv', 'practicuum/shared/data/target_qsar/hmgcr_dataset.json', 'practicuum/shared/data/target_qsar/hmgcr_result.json', 'practicuum/shared/data/target_qsar/manifest.json']))


C:\Users\Miakhlov\Documents\GitHub\Computational chemistry_course\tmp\reasoning-full-20261120\workspace\my_research\evidence\M12-L02\20261120T105441520386Z-A-dda82c2032a3442d8fb631341255f62e.json


# M12. Статья, в которой лучший метод зависит от задачи

Модель A2A уже дала прогноз, но её химическая область требует review. Перед решением по лекарству проверим более общее утверждение: достаточно ли назвать модель графовой, чтобы считать её лучше? Читаем **Wu et al., MoleculeNet (2018), DOI 10.1039/C7SC02664A**, разделы о разбиении/метриках и **Table 3**. [Первичная статья](https://doi.org/10.1039/C7SC02664A), [доступный структурированный текст NCBI](https://www.ncbi.nlm.nih.gov/research/bionlp/RESTful/pmcoa.cgi/BioC_xml/PMC5868307/unicode).

Локально сохранены три выбранные фактические строки, не полный текст и не вся таблица. [Паспорт и значения](../../../../../shared/data/article_reading/moleculenet_selected_results.json) содержат источник, Table ID и SHA исходного локального cache. В самой сводной строке нет всех определений target и неопределённости; это нужно проверить в полном тексте. Права статьи не меняются.

In [7]:
import json
from course_support import DATA,workspace,write_json
article=json.loads((DATA/'article_reading/moleculenet_selected_results.json').read_text(encoding='utf-8'))
benchmark_audit=[]
for row in article['rows']:
    # Знак улучшения зависит от метрики: для ошибки меньше лучше, для AUC больше лучше.
    c=row['conventional_value'];g=row['graph_value']
    benefit=(g-c) if row['higher_is_better'] else (c-g)
    benchmark_audit.append({'dataset':row['dataset'],'metric':row['metric'],
        'graph_directional_benefit':benefit,'graph_better_in_this_row':benefit>0,
        'relative_error_reduction':None if row['higher_is_better'] else benefit/c})
display(benchmark_audit)
write_json(workspace('M12-L02','case')/'article_benchmark_audit.json',
           {'kind':'calculated','scope':'three published benchmark summary rows; no new training',
            'doi':article['doi'],'comparisons':benchmark_audit})


[{'dataset': 'ESOL',
  'metric': 'RMSE',
  'graph_directional_benefit': 0.41000000000000003,
  'graph_better_in_this_row': True,
  'relative_error_reduction': 0.4141414141414142},
 {'dataset': 'PDBbind (full)',
  'metric': 'RMSE',
  'graph_directional_benefit': -0.18999999999999995,
  'graph_better_in_this_row': False,
  'relative_error_reduction': -0.15199999999999997},
 {'dataset': 'BBBP',
  'metric': 'AUC-ROC',
  'graph_directional_benefit': -0.039000000000000035,
  'graph_better_in_this_row': False,
  'relative_error_reduction': None}]

На выбранных строках один победитель не распространяется на все задачи. Относительное уменьшение RMSE осмысленно внутри одной строки; процентное изменение AUC не выдаём за уменьшение физической ошибки. Три специально выбранных примера не оценивают долю побед во всём benchmark. Сводка лучших моделей также не изолирует причинный эффект архитектуры.

Безразмерность сама по себе не делает шкалы сопоставимыми. pKi = −log10(Ki / (1 mol/L)) безразмерен, как и AUROC, но первый описывает аффинность, а второй — ранжирование классов. Разность AUROC нельзя интерпретировать как ошибку pKi или величину проницаемости.

Свяжите это с A2A: значение ошибки другой задачи не является независимой проверкой pKi, а scaffold split не устраняет любой assay/time shift. Перейдите к [мини-рецензии B](../steps/03_article_review.md), сохранив различие между опубликованной метрикой, нашим пересчётом и химическим выводом.